In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import kagglehub as kh
import os

# Load dataset
path = kh.dataset_download("rabieelkharoua/alzheimers-disease-dataset")
print("Path to dataset files:", path)

files = os.listdir(path)
print("Files in directory:", files)

csv_file = [f for f in files if f.endswith('.csv')][0]
csv_path = os.path.join(path, csv_file)


df = pd.read_csv(csv_path)
print("Shape:", df.shape)
df.head()

X = df.drop(columns=['Diagnosis', 'PatientID', 'DoctorInCharge'])
y = df['Diagnosis']

# Define feature sets
clinical_markers = [
    'MMSE', 'ADL', 'FunctionalAssessment', 'MemoryComplaints',
    'BehavioralProblems', 'Disorientation', 'Confusion',
    'PersonalityChanges', 'DifficultyCompletingTasks', 'Forgetfulness'
]

X_full = X.copy()
X_reduced = X.drop(columns=clinical_markers)

# Split
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y, test_size=0.2, stratify=y, random_state=42
)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reduced, y, test_size=0.2, stratify=y, random_state=42
)

# Fit models - full baseline
lr_f = LogisticRegression(solver='liblinear', random_state=42)
lda_f = LinearDiscriminantAnalysis(solver='svd')
rf_f = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=42)

lr_f.fit(X_train_f, y_train_f)
lda_f.fit(X_train_f, y_train_f)
rf_f.fit(X_train_f, y_train_f)

# Fit models - reduced set
lr_r = LogisticRegression(solver='liblinear', random_state=42)
lda_r = LinearDiscriminantAnalysis(solver='svd')
rf_r = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=42)

lr_r.fit(X_train_r, y_train_r)
lda_r.fit(X_train_r, y_train_r)
rf_r.fit(X_train_r, y_train_r)

# Predicted probabilities
y_prob_lr_f = lr_f.predict_proba(X_test_f)[:, 1]
y_prob_lda_f = lda_f.predict_proba(X_test_f)[:, 1]
y_prob_rf_f = rf_f.predict_proba(X_test_f)[:, 1]

y_prob_lr_r = lr_r.predict_proba(X_test_r)[:, 1]
y_prob_lda_r = lda_r.predict_proba(X_test_r)[:, 1]
y_prob_rf_r = rf_r.predict_proba(X_test_r)[:, 1]

# ROC curves
fpr_lr_f, tpr_lr_f, _ = roc_curve(y_test_f, y_prob_lr_f)
fpr_lda_f, tpr_lda_f, _ = roc_curve(y_test_f, y_prob_lda_f)
fpr_rf_f, tpr_rf_f, _ = roc_curve(y_test_f, y_prob_rf_f)

fpr_lr_r, tpr_lr_r, _ = roc_curve(y_test_r, y_prob_lr_r)
fpr_lda_r, tpr_lda_r, _ = roc_curve(y_test_r, y_prob_lda_r)
fpr_rf_r, tpr_rf_r, _ = roc_curve(y_test_r, y_prob_rf_r)

auc_lr_f = auc(fpr_lr_f, tpr_lr_f)
auc_lda_f = auc(fpr_lda_f, tpr_lda_f)
auc_rf_f = auc(fpr_rf_f, tpr_rf_f)

auc_lr_r = auc(fpr_lr_r, tpr_lr_r)
auc_lda_r = auc(fpr_lda_r, tpr_lda_r)
auc_rf_r = auc(fpr_rf_r, tpr_rf_r)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))

# Full baseline - solid lines
ax.plot(fpr_lr_f, tpr_lr_f, color='steelblue', linewidth=2, linestyle='-',
        label=f'LR  - Full     (AUC = {auc_lr_f:.3f})')
ax.plot(fpr_lda_f, tpr_lda_f, color='darkorange', linewidth=2, linestyle='-',
        label=f'LDA - Full     (AUC = {auc_lda_f:.3f})')
ax.plot(fpr_rf_f, tpr_rf_f, color='seagreen', linewidth=2, linestyle='-',
        label=f'RF  - Full     (AUC = {auc_rf_f:.3f})')

# Reduced set - dashed lines, same colors
ax.plot(fpr_lr_r, tpr_lr_r, color='steelblue', linewidth=1.5, linestyle='--',
        label=f'LR  - Reduced (AUC = {auc_lr_r:.3f})')
ax.plot(fpr_lda_r, tpr_lda_r, color='darkorange', linewidth=1.5, linestyle='--',
        label=f'LDA - Reduced (AUC = {auc_lda_r:.3f})')
ax.plot(fpr_rf_r, tpr_rf_r, color='seagreen', linewidth=1.5, linestyle='--',
        label=f'RF  - Reduced (AUC = {auc_rf_r:.3f})')

# Random guessing baseline
ax.plot([0, 1], [0, 1], color='gray', linestyle=':', linewidth=1.2,
        label='Random Guessing')

ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('Comparative ROC Analysis: Full Baseline vs. Reduced Feature Set', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Full baseline:")
print(f"  LR:  {auc_lr_f:.4f} | LDA: {auc_lda_f:.4f} | RF: {auc_rf_f:.4f}")
print("Reduced set:")
print(f"  LR:  {auc_lr_r:.4f} | LDA: {auc_lda_r:.4f} | RF: {auc_rf_r:.4f}")